# Image statistics

AstroVIPER provides three access points to image statistics:

1. **Node-task API** for an in-memory `DataArray`/`Dataset` or one on-disk image. This is the most convenient interactive interface.
2. **Distributed application API** for a partitioned on-disk XRADIO/Zarr image.
3. **Processing-function API** for already selected, NumPy-backed arrays and explicit state creation/merging.

Every interface follows the same order: select pixels, apply masks and value filters, reduce the requested axes, and return only the requested statistics. Unreduced dimensions and their coordinates remain in the result.

In [ ]:
import numpy as np
import xarray as xr

from astroviper.distributed_applications.image_analysis import (
    image_statistics as distributed_image_statistics,
)
from astroviper.node_tasks.image_analysis import (
    build_image_selection,
)
from astroviper.node_tasks.image_analysis import (
    image_statistics as node_image_statistics,
)
from astroviper.processing_functions.image_analysis.statistics import (
    create_statistics_state,
    finalize_statistics_state,
    merge_statistics_states,
)

## Example image cube

The examples use a small five-dimensional image with physical coordinates, units, NaNs, and a two-dimensional spatial mask.

In [ ]:
random = np.random.default_rng(7)
values = random.normal(size=(2, 4, 2, 5, 6))
values[0, 0, 0, 0, 0] = np.nan
values[1, 3, 1, 4, 5] = 12.0

image = xr.DataArray(
    values,
    dims=("time", "frequency", "polarization", "l", "m"),
    coords={
        "time": [0.0, 10.0],
        "frequency": [100e9, 101e9, 102e9, 103e9],
        "polarization": ["I", "Q"],
        "l": np.arange(5),
        "m": np.arange(6),
    },
    attrs={"units": "Jy/beam"},
    name="SKY",
)
spatial_mask = xr.DataArray(
    np.indices((5, 6)).sum(axis=0) % 2 == 0,
    dims=("l", "m"),
)
image

## 1. Direct node-task access

With no `axes` argument, every dimension is reduced. The default statistic subset is `max`, `min`, `sum`, `mean`, and `npts`.

In [ ]:
global_default = node_image_statistics(image)
global_default

### Requesting a statistic subset

Only names listed in `statistics` appear in the returned dataset. Exact median and median absolute deviation retain samples internally; compact statistics do not.

In [ ]:
robust_and_peak = node_image_statistics(
    image,
    statistics=("median", "medabsdevmed", "rms", "max", "maxpos"),
)
robust_and_peak

### Reducing selected axes

Reducing only `l` and `m` produces one result per time, frequency, and polarization plane. Position variables add `statistics_axis`, whose values identify the reduced axes.

In [ ]:
per_plane = node_image_statistics(
    image,
    axes=("l", "m"),
    statistics=("mean", "sigma", "min", "minpos", "max", "maxpos"),
)
per_plane

### Channels, time, polarization, and spatial boxes

`chans` and `timerange` accept scalars, inclusive ranges (`a~b`), stepped ranges (`a~b^step`), comparisons, and comma/semicolon unions. Regular selections stay as slices; irregular unions use integer arrays. Polarization uses labels. Boxes use inclusive `l0,m0,l1,m1` pixel coordinates.

In [ ]:
selected_result = node_image_statistics(
    image,
    axes=("l", "m"),
    chans="0,2~3",  # irregular channel union
    timerange="1",  # retain the second time plane
    stokes="I",  # retain one polarization plane
    box="1,1,4,5",  # inclusive spatial subimage
    statistics=("mean", "npts", "max", "maxpos"),
)
selected_result

In [ ]:
regular = build_image_selection(image, chans="0~3")
irregular = build_image_selection(image, chans="0,2~3")
regular.effective_indexers["frequency"], irregular.effective_indexers["frequency"]

### Multiple spatial boxes

Additional groups of four coordinates form a union. The node loads their common bounding rectangle and masks pixels in the gaps.

In [ ]:
two_boxes = node_image_statistics(
    image,
    box="0,0,1,1,3,4,4,5",
    statistics=("mean", "sum", "npts"),
)
two_boxes

### External and named masks

An external mask can have fewer dimensions than the image. Set `stretch=True` to broadcast it. A Dataset can instead contain a named mask variable. Mask value `True` means include the pixel.

In [ ]:
externally_masked = node_image_statistics(
    image,
    axes=("l", "m"),
    mask=spatial_mask,
    stretch=True,
    statistics=("mean", "median", "npts"),
)
externally_masked

In [ ]:
image_dataset = xr.Dataset({"SKY": image, "MASK_SKY": spatial_mask})
named_mask_result = node_image_statistics(
    image_dataset,
    data_variable="SKY",
    axes=("l", "m"),
    mask="MASK_SKY",
    stretch=True,
    statistics=("rms", "sigma", "npts"),
)
named_mask_result

### Pixel-value filters

`includepix=(low, high)` retains values inside an inclusive interval. `excludepix` removes values inside its interval. Filters run after spatial selection and Boolean masking.

In [ ]:
filtered = node_image_statistics(
    image,
    includepix=(-1.0, 1.0),
    statistics=("min", "max", "mean", "npts"),
)
filtered

## 2. Processing-function access

Use this interface when data is already selected and loaded. It separates state creation from finalization and is useful for custom pipelines or explicit distributed reductions.

In [ ]:
loaded_selection = image.isel(frequency=slice(1, 3)).load()
state = create_statistics_state(
    loaded_selection,
    dims=("l", "m"),
    statistics=("median", "medabsdevmed"),
    positions={"l": slice(0, 5, 1), "m": slice(0, 6, 1)},
)
processing_result = finalize_statistics_state(
    state,
    statistics=("mean", "median", "medabsdevmed", "maxpos"),
)
processing_result

### Explicitly merging partial states

The following example partitions frequency, reduces frequency locally, and merges the two partial states. Absolute position slices ensure `maxpos` refers to the original cube.

In [ ]:
partial_states = []
for start, stop in ((0, 2), (2, 4)):
    partial = image.isel(frequency=slice(start, stop)).load()
    partial_states.append(
        create_statistics_state(
            partial,
            dims="frequency",
            positions={"frequency": slice(start, stop, 1)},
        )
    )
merged_state = merge_statistics_states(
    partial_states,
    partition_dim="frequency",
    reduction_dims=("frequency",),
)
merged_result = finalize_statistics_state(
    merged_state, statistics=("mean", "max", "maxpos", "npts")
)
merged_result

## 3. Distributed on-disk access

The distributed application requires an on-disk XRADIO/Zarr image and builds a GraphVIPER map/reduce workflow. Its scientific selection arguments match the node API. A distributed mask must be the name of a variable in the store. Set the path and enable the cell when running with an available image. This option is only recommended for very large images that are on disk.

In [ ]:
RUN_DISTRIBUTED_EXAMPLE = False
image_store = "/path/to/image.zarr"

if RUN_DISTRIBUTED_EXAMPLE:
    distributed_result = distributed_image_statistics(
        image_store,
        data_variable="SKY",
        axes=("time", "polarization", "l", "m"),
        chans="0~31",
        mask="MASK_SKY",
        stretch=True,
        statistics=("mean", "rms", "sigma", "max", "maxpos", "npts"),
        partition_dim="frequency",
        n_partitions=8,
        reduce_mode="tree",
    )
    display(distributed_result)

## Result and performance notes

- Value statistics preserve the input variable's units. `npts` and extrema positions are unitless.
- `minpos` and `maxpos` contain absolute zero-based source-image positions along `statistics_axis`.
- Empty reductions use NaN for numerical values, zero for `npts`, and `-1` for positions.
- Regular selectors and worker partitions remain slices through lazy `isel`; genuinely irregular unions use integer arrays.
- Most distributed statistics use a compact mergeable state. Exact `median` and `medabsdevmed`/`mad` retain selected samples and therefore require approximately input-sized state memory and transfer.
- Multiple boxes load their bounding rectangle, so widely separated boxes can increase I/O.